# .invoke() (the single shot)
## Sends one question and waits for one answer to come back all at once. Like throwing a ball and waiting for it to be thrown back.

In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time
load_dotenv()

True

In [11]:
llm=ChatGroq(model="llama-3.1-8b-instant")
parser=StrOutputParser()
base_prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant. Be concise, max 3 sentences."),
    ("human","{question}")])
chain=base_prompt|llm|parser


In [12]:
start = time.time()
result = chain.invoke({"question": "What is LangChain in one sentence?"})
end = time.time()

In [13]:
print(f"Answer : {result}")
print(f"Time   : {end - start:.2f} seconds")
# Notice: Python waited the full time before printing anything
print()

Answer : LangChain is an open-source Python framework that enables the development and deployment of large language models (LLMs) as scalable, modular, and extensible AI applications. It provides a unified interface for integrating various LLMs, data storage, and external systems, allowing developers to create complex AI workflows. LangChain is designed to simplify the building, testing, and deployment of AI applications.
Time   : 0.91 seconds



# .batch() (the multi-pack)
## Sends a list of many questions at the exact same time and gets a list of answers back

In [14]:
questions = [
    {"question": "What is RAG in one sentence?"},
    {"question": "What is a vector database in one sentence?"},
    {"question": "What is LangGraph in one sentence?"},
    {"question": "What is fine-tuning in one sentence?"},
    {"question": "What is a prompt template in one sentence?"},
]

# Time the batch call
start = time.time()
results = chain.batch(questions)
end = time.time()

print(f"Answered {len(questions)} questions in {end - start:.2f} seconds")
print()
for q, r in zip(questions, results):
    print(f"Q: {q['question']}")
    print(f"A: {r}")
    print()


Answered 5 questions in 0.56 seconds

Q: What is RAG in one sentence?
A: RAG (Red, Amber, Green) is a traffic light system used to track and manage tasks, projects, or workflows, where Red indicates issues or delays, Amber signifies caution or attention required, and Green signifies completion or progress.

Q: What is a vector database in one sentence?
A: A vector database is a type of database that stores and retrieves data as numerical vectors, often used for similarity searches and machine learning tasks, such as nearest neighbor searches and clustering.

Q: What is LangGraph in one sentence?
A: LangGraph is a graph-based language model that uses graph neural networks to represent and process linguistic structures, allowing for more accurate and nuanced language understanding.

Q: What is fine-tuning in one sentence?
A: Fine-tuning refers to the process of adjusting a pre-trained artificial intelligence or machine learning model to a specific task or domain by retraining it on a sma

In [15]:
print("── Now compare: same questions with a loop ──")
start = time.time()
for q in questions:
    r = chain.invoke(q)
end = time.time()
print(f"Loop took: {end - start:.2f} seconds")
print(f"Batch is roughly {((end-start)/((end-start)/len(questions))):.0f}x faster")
print()

── Now compare: same questions with a loop ──
Loop took: 2.03 seconds
Batch is roughly 5x faster



# .stream() (The Live Ticker)
## Spits out the answer word-by-word as the AI thinks of it, instead of waiting for the whole paragraph. Like watching text appear live on a screen.

In [16]:
stream_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a storyteller. Write vividly."),
    ("human", "Write a 4-sentence story about {hero}.")
])
stream_chain = stream_prompt | llm | parser

print("Watch the words appear one by one:")
print()

full_response = ""
for chunk in stream_chain.stream({"hero": "a developer who learned AI"}):
    print(chunk, end="", flush=True)
    # end=""     → no newline between chunks, they join together
    # flush=True → force print immediately, don't buffer
    full_response += chunk

print()
print()
print(f"Total characters received: {len(full_response)}")
print()

Watch the words appear one by one:

As the sun dipped into the horizon, casting a warm orange glow over the city, Ethan sat hunched over his computer, his fingers flying across the keyboard as he delved deeper into the mysteries of artificial intelligence. The once-novice developer had spent countless hours devouring tutorials, attending webinars, and experimenting with code, and the fruits of his labor were finally beginning to bear fruit - a sophisticated AI model that could learn and adapt with uncanny speed. With each passing day, Ethan's machine grew smarter, its capabilities expanding exponentially as it devoured vast amounts of data and refined its understanding of the world. As the night wore on, Ethan leaned back in his chair, a sense of awe and wonder washing over him as he beheld the digital creature he had brought into being.

Total characters received: 813



# Sequential Chain (The Relay Race)
## Connects multiple AI steps together where the output of the first step automatically becomes the input for the next step.

In [17]:
prompt1=ChatPromptTemplate.from_messages([("system", "You invent cool superhero names for animals. Respond with JUST the name."),
    ("human", "Make a superhero name for a {animal}.")
])
chain1=prompt1|llm|parser
prompt2=ChatPromptTemplate.from_messages([("system", "You write funny, short catchphrases for superheroes."),
    ("human", "Write a catchphrase for a superhero named {hero_name}.")
])
chain2=prompt2|llm|parser
sequential_chain = (
    {"hero_name": chain1}  # Step 1: Run chain1 to get the 'hero_name'
    | chain2               # Step 2: Pass that 'hero_name' directly into chain2
)
# Run the whole relay race at once!
result = sequential_chain.invoke({"animal": "Hedgehog"})

print(result)

"Shielding the world, one quill at a time."
